# Housing Application Approval

## Dataset

We use the ACSMobility dataset from the Folktables benchmark, derived from the U.S. Census American Community Survey (ACS).

The task is a binary classification problem that predicts whether a person changed residence in the past year (1) or did not move (0). Residential mobility is commonly used as a proxy for housing stability, since frequent or involuntary moves are often associated with housing insecurity, evictions, and barriers to housing approval.

The input features include demographic and socioeconomic variables such as age, education, marital status, employment characteristics, and income-related attributes.

The dataset also contains sensitive attributes — race and sex —, which are used to evaluate fairness in housing-related decision-making.

## Race (RAC1P)

The variable RAC1P represents self-identified race in the ACS data.

- 1: White alone
- 2: Black or African American alone
- 3: American Indian or Alaska Native alone
- 4: Alaska Native alone
- 5: American Indian alone
- 6: Asian alone
- 7: Native Hawaiian or Other Pacific Islander alone
- 8: Some other race alone
- 9: Two or more races

### Sex (SEX)

The variable SEX is binary in the ACS:

- 1: Male
- 2: Female

## Models

We train two logistic regression models on the ACSMobility dataset.

### Accuracy-first model

This model uses all available features, including sensitive attributes such as race and sex, and is optimized solely for predictive performance (accuracy and ROC AUC). It represents a standard automated machine learning pipeline that prioritizes accuracy without considering fairness constraints.

### Fairness-aware model

This model excludes race and sex from the input feature set to reduce direct discrimination. While this may lead to a modest reduction in predictive performance, it aims to decrease disparities in predicted housing instability across protected groups.

Both models are evaluated using standard performance metrics as well as fairness metrics, including:
Demographic Parity, which measures differences in predicted probabilities across groups
Equal Opportunity, which measures differences in true positive rates across race and sex
By comparing these two models, we analyze the trade-off between accuracy and fairness in automated housing-related decision systems.

## Data download

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
from folktables import ACSDataSource, ACSMobility


# Load ACS data
data_source = ACSDataSource(survey_year="2018", horizon="1-Year", survey="person")
acs_data = data_source.get_data(states=["CA"], download=True)

# Define task
task = ACSMobility

# Get filtered dataframe used by the task
acs_df, y, _ = task.df_to_pandas(acs_data)

# Build modeling dataframe
df = acs_df[task.features].copy()
df["target"] = y

# Add sensitive attributes
df["race"] = acs_df["RAC1P"].to_numpy()
df["sex"]  = acs_df["SEX"].to_numpy()

print(df.shape)
print(df[["race","sex","target"]].head())

# Define feature columns
feature_names = [c for c in df.columns if c not in ["target", "race", "sex"]]

X = df[feature_names]
y = df["target"].astype(int)

(80329, 24)
   race  sex  target
0   8.0  1.0   False
1   1.0  1.0    True
2   1.0  2.0    True
3   6.0  1.0   False
4   1.0  1.0    True
1    61478
0    18851
Name: target, dtype: int64


## Train/Test split

In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Model 1: Accuracy-first

In [3]:
model_acc = LogisticRegression(max_iter=2000)
model_acc.fit(X_train_s, y_train)

y_pred = model_acc.predict(X_test_s)
y_prob = model_acc.predict_proba(X_test_s)[:, 1]

print("Accuracy (Housing baseline):", accuracy_score(y_test, y_pred))
print("ROC AUC (Housing baseline):", roc_auc_score(y_test, y_prob))

Accuracy (Housing baseline): 0.7662558612390555
ROC AUC (Housing baseline): 0.6583981266877864


## Fairness (baseline)

In [4]:
test_df = df.loc[X_test.index, ["race", "sex"]].copy()
test_df["prob"] = y_prob

print("\nAvg predicted housing instability by race:")
print(test_df.groupby("race")["prob"].mean())

print("\nAvg predicted housing instability by sex:")
print(test_df.groupby("sex")["prob"].mean())

race_gap = test_df.groupby("race")["prob"].mean().max() - test_df.groupby("race")["prob"].mean().min()
sex_gap  = test_df.groupby("sex")["prob"].mean().max()  - test_df.groupby("sex")["prob"].mean().min()

print("\nDemographic Parity gap (race):", race_gap)
print("Demographic Parity gap (sex):", sex_gap)


Avg predicted housing instability by race:
race
1.0    0.757438
2.0    0.747875
3.0    0.775632
5.0    0.743930
6.0    0.755815
7.0    0.771702
8.0    0.807744
9.0    0.784587
Name: prob, dtype: float64

Avg predicted housing instability by sex:
sex
1.0    0.764158
2.0    0.767617
Name: prob, dtype: float64

Demographic Parity gap (race): 0.06381434507456951
Demographic Parity gap (sex): 0.0034596847782091267


# Model 2: Fairness-aware model

In [5]:
fair_features = [f for f in feature_names if f not in ["RAC1P", "SEX"]]

X_fair = df[fair_features]
y = df["target"].astype(int)

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_fair, y, test_size=0.3, random_state=42, stratify=y
)

scaler_f = StandardScaler()
X_train_f_s = scaler_f.fit_transform(X_train_f)
X_test_f_s = scaler_f.transform(X_test_f)

model_fair = LogisticRegression(max_iter=2000)
model_fair.fit(X_train_f_s, y_train_f)

y_prob_f = model_fair.predict_proba(X_test_f_s)[:, 1]

## Fairness-aware evaluation

In [6]:
test_df_f = df.loc[X_test_f.index, ["race", "sex"]].copy()
test_df_f["prob"] = y_prob_f

print("\nAvg predicted housing instability by race (fairness-aware):")
print(test_df_f.groupby("race")["prob"].mean())

print("\nAvg predicted housing instability by sex (fairness-aware):")
print(test_df_f.groupby("sex")["prob"].mean())

race_gap_f = test_df_f.groupby("race")["prob"].mean().max() - test_df_f.groupby("race")["prob"].mean().min()
sex_gap_f  = test_df_f.groupby("sex")["prob"].mean().max()  - test_df_f.groupby("sex")["prob"].mean().min()

print("\nDemographic Parity gap (race, fairness-aware):", race_gap_f)
print("Demographic Parity gap (sex, fairness-aware):", sex_gap_f)


Avg predicted housing instability by race (fairness-aware):
race
1.0    0.765768
2.0    0.752223
3.0    0.776673
5.0    0.736558
6.0    0.747271
7.0    0.757448
8.0    0.792592
9.0    0.763459
Name: prob, dtype: float64

Avg predicted housing instability by sex (fairness-aware):
sex
1.0    0.762117
2.0    0.770137
Name: prob, dtype: float64

Demographic Parity gap (race, fairness-aware): 0.05603401501019645
Demographic Parity gap (sex, fairness-aware): 0.008020138840541535
